[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/08_tag_11_12_lstm_gates_untersuchen.ipynb)

# Tag 11 & 12 – 08: Wie eine LSTM lernt, was sie behalten soll

Eine LSTM erhält Zahlenfolgen mit zwei Markierungen: **Speichern** kennzeichnet den relevanten Wert und **Ausgeben** fordert diesen Wert später wieder an. Dazwischen stehen zufällige Ablenkungswerte, die teilweise deutlich größer sind als die zu merkende Zahl.

Das Modell bekommt keine Regeln für seine Signale. Es sieht nur Eingaben und richtige Zielausgaben. Gradient Descent lernt daraus die Gewichte für Input-Signal $i_t$, Forget-Signal $f_t$, neue Information $g_t$ und Output $o_t$.

## Die zentrale Frage

Ein Signal reagiert nicht automatisch auf große Zahlen. Zuerst entsteht eine **gewichtete Summe** aus der Eingabe $x_t$, dem vorherigen Hidden State $h_{t-1}$ und einem Bias. Eine Sigmoid-Funktion wandelt diese Summe anschließend in einen Wert zwischen 0 und 1 um.

$$z_i = x_t W_i + h_{t-1} U_i + b_i \qquad i_t = \sigma(z_i)$$

Damit die Bezeichnungen genau zur Folie passen, verwenden wir im gesamten Notebook dieselbe Zuordnung:

- **Eingabe:** $x_t$
- **vorheriger Hidden State:** $h_{t-1}$
- **vorheriger Zellzustand:** $c_{t-1}$
- **Forget-Signal:** $f_t$; das Produkt $f_t \odot c_{t-1}$ heißt hier **Behaltebeitrag**
- **Input-Signal:** $i_t$ und **neue Information:** $g_t$; das Produkt $i_t \odot g_t$ heißt hier **Schreibbeitrag**
- **Output:** $o_t$
- **Vorhersage:** $y_t$

Der vollständige Datenfluss lautet:

$$\begin{aligned} f_t &= \sigma(x_tW_f+h_{t-1}U_f+b_f) \\ i_t &= \sigma(x_tW_i+h_{t-1}U_i+b_i) \\ g_t &= \tanh(x_tW_g+h_{t-1}U_g+b_g) \\ c_t &= \underbrace{f_t \odot c_{t-1}}_{\text{Behaltebeitrag}} + \underbrace{i_t \odot g_t}_{\text{Schreibbeitrag}} \\ o_t &= \sigma(x_tW_o+h_{t-1}U_o+b_o) \\ h_t &= o_t \odot \tanh(c_t) \\ y_t &= h_tW_y+b_y \end{aligned}$$

Das Zeichen $\odot$ bezeichnet eine **elementweise Multiplikation**. Weil dieses Beispiel nur eine LSTM-Einheit besitzt, ist sie hier einfach die Multiplikation zweier Zahlen. Welche Eingabe die Signale steuert, hängt von den **gelernten Gewichten** ab. In diesem Beispiel soll nicht die größte Zahl gespeichert werden, sondern die markierte Zahl – auch wenn sie nur 0,05 beträgt.

In [ ]:
import time                         # Trainingsdauer messen
import matplotlib.pyplot as plt    # Daten, Training und Signalverläufe zeichnen
import numpy as np                 # Sequenzen und manuelle LSTM-Rechnung erzeugen
import pandas as pd                # Gewichte und Zwischenergebnisse tabellarisch darstellen
import tensorflow as tf            # LSTM und GradientTape verwenden
from IPython.display import display, Markdown  # Tabellen und erklärende Zwischenüberschriften anzeigen

plt.style.use('seaborn-v0_8-whitegrid')  # Einheitlichen Diagrammstil festlegen
RANDOM_STATE = 83                         # Reproduzierbare Daten und Startgewichte verwenden
np.random.seed(RANDOM_STATE)               # NumPy-Zufallsgenerator initialisieren
tf.keras.utils.set_random_seed(RANDOM_STATE)  # TensorFlow-Zufallsgenerator initialisieren
tf.get_logger().setLevel('ERROR')          # Unwichtige TensorFlow-Statusmeldungen ausblenden

## Trainingsdaten: markierte Information statt größter Wert

Jeder Zeitschritt besitzt drei Eingabemerkmale:

- **Wert:** eine beliebige Zahl zwischen −1 und 1,
- **Speichern:** genau an einer frühen Position gleich 1,
- **Ausgeben:** am letzten Zeitschritt gleich 1.

Nur am letzten Schritt lautet die Zielausgabe gleich dem zuvor markierten Wert. An allen anderen Positionen ist die Zielausgabe null. Der Speicherzeitpunkt und alle Werte werden für jede Sequenz neu ausgelost.

In [ ]:
SEQUENZLAENGE = 12       # Anzahl der Zeitschritte pro Beispiel
ANZAHL_TRAINING = 5000   # Sequenzen für Gewichtsupdates
ANZAHL_VALIDIERUNG = 1000  # Unabhängige Sequenzen für die Modellauswahl
ANZAHL_TEST = 1000       # Erst nach dem Training verwendete Testsequenzen

def erstelle_datensatz(anzahl, seed):  # Selektive Merkaufgabe mit neuen Zufallswerten erzeugen
    rng = np.random.default_rng(seed)  # Eigenen Zufallsgenerator für diesen Datensatz anlegen
    werte = rng.uniform(-1.0, 1.0, size=(anzahl, SEQUENZLAENGE)).astype(np.float32)  # Kleine und große Zahlen mischen
    speicher_positionen = rng.integers(1, 5, size=anzahl)  # Relevante Position zwischen Schritt 1 und 4 wählen
    ausgabe_masken = np.zeros((anzahl, SEQUENZLAENGE), dtype=bool)  # Ausgabe zunächst an allen Positionen deaktivieren
    ausgabe_masken[:, -1] = True  # Genau am letzten Schritt die gespeicherte Zahl anfordern

    X = np.zeros((anzahl, SEQUENZLAENGE, 3), dtype=np.float32)  # Eingabetensor vorbereiten
    X[:, :, 0] = werte  # Erste Spalte enthält die eigentlichen Zahlen
    X[np.arange(anzahl), speicher_positionen, 1] = 1.0  # Zweite Spalte markiert die relevante Zahl
    X[:, :, 2] = ausgabe_masken  # Dritte Spalte markiert gewünschte Ausgaben

    zielwerte = werte[np.arange(anzahl), speicher_positionen]  # Markierte Zahl jeder Sequenz auslesen
    y = np.zeros((anzahl, SEQUENZLAENGE, 1), dtype=np.float32)  # Zielsequenzen zunächst auf null setzen
    y[:, :, 0] = ausgabe_masken * zielwerte[:, None]  # Nur an Abfragepositionen den gespeicherten Wert verlangen
    return X, y, zielwerte, speicher_positionen  # Eingaben, Zielsequenzen und Metadaten zurückgeben

X_train, y_train, ziele_train, speicher_train = erstelle_datensatz(ANZAHL_TRAINING, RANDOM_STATE)  # Trainingsdaten erzeugen
X_valid, y_valid, ziele_valid, speicher_valid = erstelle_datensatz(ANZAHL_VALIDIERUNG, RANDOM_STATE + 1)  # Validierungsdaten erzeugen
X_test, y_test, ziele_test, speicher_test = erstelle_datensatz(ANZAHL_TEST, RANDOM_STATE + 2)  # Testdaten erzeugen

betragsziele = np.abs(ziele_train)  # Zielgrößen für anschauliche Beispiele bestimmen
beispiele = [np.argmin(np.abs(betragsziele - grenze)) for grenze in [0.06, 0.45, 0.90]]  # Kleines, mittleres und großes Ziel auswählen
fig, achsen = plt.subplots(3, 1, figsize=(12, 8), sharex=True)  # Drei Trainingssequenzen untereinander zeichnen
for achse, nummer in zip(achsen, beispiele):  # Jedes ausgewählte Beispiel darstellen
    positionen = np.arange(SEQUENZLAENGE)  # Diskrete Zeitpositionen erzeugen
    achse.plot(positionen, X_train[nummer, :, 0], 'o-', color='0.45', label='Eingabewerte')  # Alle Zahlen der Sequenz zeichnen
    speicher_t = speicher_train[nummer]  # Relevante Position dieses Beispiels auslesen
    ausgabe_t = np.flatnonzero(X_train[nummer, :, 2])  # Alle Abfragepositionen bestimmen
    achse.scatter(speicher_t, X_train[nummer, speicher_t, 0], s=140, color='tab:blue', zorder=4, label='zu speichern')  # Relevante Zahl hervorheben
    achse.scatter(ausgabe_t, y_train[nummer, ausgabe_t, 0], marker='X', s=90, color='tab:green', zorder=4, label='Zielausgabe')  # Geforderte Ausgaben markieren
    achse.axhline(ziele_train[nummer], color='tab:green', linestyle=':', alpha=0.7)  # Gespeicherten Sollwert als Linie zeigen
    achse.set_ylabel(f'Ziel {ziele_train[nummer]:.2f}')  # Größe der markierten Information beschriften
achsen[0].set_title('Die markierte Zahl zählt – nicht die größte Zahl der Sequenz')  # Aussage der Rohdatenabbildung nennen
achsen[0].legend(loc='upper right')  # Legende einmal anzeigen
achsen[-1].set_xlabel('Zeitschritt')  # Gemeinsame Zeitachse beschriften
achsen[-1].set_xticks(np.arange(SEQUENZLAENGE))  # Jeden Zeitschritt anzeigen
plt.tight_layout()  # Abstände anpassen
plt.show()  # Trainingsbeispiele ausgeben

print('X_train:', X_train.shape, '= (Sequenzen, Zeitschritte, Merkmale)')  # Eingabeform dokumentieren
print('y_train:', y_train.shape, '= eine Zielausgabe pro Zeitschritt')  # Ausgabeform dokumentieren

## Ein minimales trainierbares LSTM-Modell

Die LSTM besitzt bewusst nur **eine Einheit**. Dadurch sind Zellzustand, Hidden State sowie $i_t$, $f_t$, $g_t$ und $o_t$ einzelne Zahlen und lassen sich später vollständig nachrechnen. Die LSTM liefert für jeden Zeitschritt einen Hidden State. Ein Dense-Layer wandelt jeden dieser States in die Vorhersage $y_t$ um.

Trotz nur einer LSTM-Einheit werden 22 Parameter gelernt: Eingabegewichte, rekurrente Gewichte, Bias-Werte sowie Gewicht und Bias des Dense-Layers.

In [ ]:
eingabe = tf.keras.Input(shape=(SEQUENZLAENGE, 3), name='sequenz')  # Zwölf Schritte mit je drei Merkmalen erwarten
lstm_schicht = tf.keras.layers.LSTM(1, return_sequences=True, name='lstm_einheit')  # Nach jedem Schritt einen skalaren Hidden State liefern
hidden_folge = lstm_schicht(eingabe)  # Eingabesequenz rekurrent verarbeiten
ausgabe_schicht = tf.keras.layers.Dense(1, name='ausgabe')  # Jeden Hidden State auf einen Vorhersagewert abbilden
vorhersage_folge = ausgabe_schicht(hidden_folge)  # Für jeden Zeitschritt die Vorhersage y_t erzeugen
modell = tf.keras.Model(eingabe, vorhersage_folge, name='selektives_lstm_gedaechtnis')  # Vollständiges Modell zusammenbauen

start_gewichte = [gewicht.copy() for gewicht in lstm_schicht.get_weights()]  # Zufällige LSTM-Startgewichte vor dem Lernen sichern
modell.summary()  # Schichten, Tensorformen und Parameterzahl ausgeben

## Wie beginnt das Lernen?

Für einen Batch berechnet das Modell zunächst Vorhersagen $y_t$ und den Fehler. Das letzte und alle weiteren markierten Ausgabeereignisse erhalten im Loss ein höheres Gewicht. GradientTape bestimmt anschließend für **jedes Gewicht**, in welche Richtung es verändert werden muss.

Die letzte Zielausgabe erhält im Loss ein höheres Gewicht als die elf vorherigen Nullen. Die erste Tabelle ist **keine Speichertabelle**: Sie zeigt die Gradienten der zwölf direkten Verbindungen von den drei Eingabemerkmalen zu den vier LSTM-Blöcken. Ein Gradient beschreibt, wie empfindlich der Fehler auf ein Gewicht reagiert. Die zweite Tabelle verfolgt fünf dieser Gewichte durch das erste Adam-Update. Beide Tabellen erklären also, **wie das Lernen beginnt** – noch nicht, wo eine konkrete Information gespeichert wird.

In [ ]:
BATCH_GROESSE = 128  # Anzahl Sequenzen pro Gewichtsupdate
LERNRATE = 0.008     # Schrittweite des Adam-Optimierers
EPOCHEN = 70         # Vollständige Trainingsdurchläufe
ZEITGEWICHTE = tf.constant([1.0] * (SEQUENZLAENGE - 1) + [12.0], dtype=tf.float32)[None, :, None]  # Finale Gedächtnisabfrage stärker gewichten

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))  # Trainingsarrays in einen TensorFlow-Datensatz umwandeln
train_ds = train_ds.shuffle(ANZAHL_TRAINING, seed=RANDOM_STATE).batch(BATCH_GROESSE)  # Sequenzen mischen und bündeln
optimierer = tf.keras.optimizers.Adam(learning_rate=LERNRATE)  # Adam für alle Gewichtsanpassungen verwenden

def gewichteter_mse(ziel, vorhersage):  # Finale Vorhersage im Trainingsfehler stärker berücksichtigen
    quadratische_fehler = tf.square(vorhersage - ziel) * ZEITGEWICHTE  # Jeden Zeitschritt mit seinem festen Gewicht multiplizieren
    batch_anzahl = tf.cast(tf.shape(ziel)[0], tf.float32)  # Aktuelle Anzahl Sequenzen im Batch bestimmen
    return tf.reduce_sum(quadratische_fehler) / (batch_anzahl * tf.reduce_sum(ZEITGEWICHTE))  # Gewichteten mittleren Fehler berechnen

erster_X_batch, erster_y_batch = next(iter(train_ds))  # Einen echten Trainingsbatch auswählen
kernel_vorher = lstm_schicht.get_weights()[0].copy()  # Eingabegewichte vor dem ersten Update sichern
with tf.GradientTape() as tape:  # Rechenweg für Backpropagation aufzeichnen
    erste_vorhersagen = modell(erster_X_batch, training=True)  # Vorwärtsdurchlauf mit zufälligen Startgewichten ausführen
    erster_loss = gewichteter_mse(erster_y_batch, erste_vorhersagen)  # Fehler des ersten Batches berechnen
gradienten = tape.gradient(erster_loss, modell.trainable_variables)  # Ableitungen nach allen 22 Parametern bestimmen
erste_kernel_gradienten = gradienten[0].numpy().copy()  # Gradienten der Eingabegewichte der vier LSTM-Blöcke sichern
optimierer.apply_gradients(zip(gradienten, modell.trainable_variables))  # Erstes tatsächliches Gewichtsupdate anwenden
kernel_nachher = lstm_schicht.get_weights()[0].copy()  # Gewichte direkt nach dem Update auslesen

block_namen = ['Input-Signal i_t', 'Forget-Signal f_t', 'Neue Information g_t', 'Output o_t']  # Keras-Reihenfolge der vier Blöcke in der Benennung der Folie festhalten
merkmal_namen = ['Wert', 'Speichern', 'Ausgeben']  # Zeilennamen der Eingabematrix definieren
display(Markdown(
    '### Gradiententabelle: Welches Gewicht erhält ein Lernsignal?\n\n'
    'Die **Zeilen** sind die drei Merkmale von $x_t$, die **Spalten** sind die vier LSTM-Blöcke. '
    'Ein großer Betrag bedeutet: Dieses Gewicht beeinflusst den aktuellen Fehler deutlich. Das Vorzeichen gibt die Richtung der Ableitung an; es bedeutet nicht direkt „Signal offen“ oder „Signal geschlossen“.'
))
display(pd.DataFrame(erste_kernel_gradienten, index=merkmal_namen, columns=block_namen).round(5))  # Gradienten des ersten Updates anzeigen

wichtige_pfade = {  # Fünf besonders gut interpretierbare Verbindungen auswählen
    'Wert → Input-Signal i_t': (0, 0),
    'Wert → neue Information g_t': (0, 2),
    'Speichern → Forget-Signal f_t': (1, 1),
    'Speichern → neue Information g_t': (1, 2),
    'Ausgeben → Output o_t': (2, 3),
}
erste_aenderungen = []  # Vorher-Nachher-Werte für die fünf Verbindungen sammeln
for name, (zeile, spalte) in wichtige_pfade.items():  # Jeden ausgewählten Gewichtspfad auslesen
    erste_aenderungen.append({
        'Verbindung': name,
        'vor Update': kernel_vorher[zeile, spalte],
        'Gradient': erste_kernel_gradienten[zeile, spalte],
        'nach Update': kernel_nachher[zeile, spalte],
    })
display(Markdown(
    '### Erstes Update: Was macht Adam aus dem Gradienten?\n\n'
    'Für fünf gut lesbare Verbindungen stehen Startgewicht, Gradient und Gewicht nach dem Update nebeneinander. Der Sinn der Tabelle ist, den Übergang von **Fehler messen** zu **Gewicht verändern** sichtbar zu machen.'
))
display(pd.DataFrame(erste_aenderungen).set_index('Verbindung').round(5))  # Erste konkrete Gewichtsanpassung zeigen
print(f'Loss des ersten Batches: {float(erster_loss):.4f}')  # Ausgangsfehler ausgeben

## Viele Updates formen die Signalregeln

Der folgende Trainingsloop wiederholt genau diesen Ablauf. Nach jeder Epoche werden Trainings- und Validierungsfehler sowie fünf wichtige Gewichte gespeichert. Das Modell mit dem besten Validierungsfehler wird anschließend wiederhergestellt.

In [ ]:
@tf.function(reduce_retracing=True)  # Wiederholten Trainingsschritt als schnellen TensorFlow-Graph ausführen
def train_schritt(X_batch, y_batch):  # Ein vollständiges Gewichtsupdate definieren
    with tf.GradientTape() as tape:  # Rechenweg dieses Batches aufzeichnen
        vorhersagen = modell(X_batch, training=True)  # Aktuelle Vorhersagen y_t berechnen
        loss = gewichteter_mse(y_batch, vorhersagen)  # Gewichteten Fehler bestimmen
    grads = tape.gradient(loss, modell.trainable_variables)  # Fehler zu allen Parametern zurückpropagieren
    optimierer.apply_gradients(zip(grads, modell.trainable_variables))  # Adam-Update anwenden
    return loss  # Batch-Loss für die Epochenstatistik zurückgeben

train_losses, valid_losses = [], []  # Fehlerverläufe vorbereiten
gewichtsverlauf = {name: [kernel_nachher[pos]] for name, pos in wichtige_pfade.items()}  # Zustand nach dem ersten Update speichern
beste_validierung = np.inf  # Bisher kleinsten Validierungsfehler initialisieren
beste_epoche = 0  # Epoche des besten Modells vorbereiten
beste_gewichte = modell.get_weights()  # Aktuellen Zustand als erste Reserve sichern
startzeit = time.perf_counter()  # Trainingszeitmessung starten

for epoche in range(1, EPOCHEN + 1):  # Mehrfach über alle Trainingssequenzen laufen
    batch_losses = []  # Fehler der aktuellen Epoche sammeln
    for X_batch, y_batch in train_ds:  # Jeden Batch genau einmal verarbeiten
        batch_losses.append(float(train_schritt(X_batch, y_batch)))  # Update ausführen und Loss merken
    train_loss = np.mean(batch_losses)  # Mittleren Trainingsfehler dieser Epoche berechnen
    valid_vorhersagen = modell(X_valid, training=False)  # Unabhängige Validierungssequenzen auswerten
    valid_loss = float(gewichteter_mse(y_valid, valid_vorhersagen))  # Validierungsfehler ohne Update bestimmen
    train_losses.append(train_loss)  # Trainingsverlauf ergänzen
    valid_losses.append(valid_loss)  # Validierungsverlauf ergänzen

    aktueller_kernel = lstm_schicht.get_weights()[0]  # Aktuelle Eingabegewichte auslesen
    for name, position in wichtige_pfade.items():  # Fünf ausgewählte Verbindungen protokollieren
        gewichtsverlauf[name].append(aktueller_kernel[position])  # Gewicht nach dieser Epoche speichern

    if valid_loss < beste_validierung:  # Nur bei einer echten Verbesserung den Modellzustand ersetzen
        beste_validierung = valid_loss  # Neuen Bestwert speichern
        beste_epoche = epoche  # Zugehörige Epoche merken
        beste_gewichte = modell.get_weights()  # Alle 22 Parameter sichern

trainingsdauer = time.perf_counter() - startzeit  # Verstrichene Trainingszeit berechnen
modell.set_weights(beste_gewichte)  # Bestes Validierungsmodell für alle folgenden Analysen wiederherstellen

fig, achsen = plt.subplots(1, 2, figsize=(14, 4.5))  # Loss und Gewichtsentwicklung nebeneinander darstellen
epochen_achse = np.arange(1, EPOCHEN + 1)  # Epochenzahlen für den Loss erzeugen
achsen[0].plot(epochen_achse, train_losses, label='Training')  # Trainingsfehler zeichnen
achsen[0].plot(epochen_achse, valid_losses, label='Validierung')  # Validierungsfehler zeichnen
achsen[0].axvline(beste_epoche, color='black', linestyle=':', label=f'bestes Modell: Epoche {beste_epoche}')  # Gewählte Epoche markieren
achsen[0].set_yscale('log')  # Deutliche Fehlerreduktion lesbar darstellen
achsen[0].set(title='Das Ausgabefeedback trainiert alle LSTM-Signale', xlabel='Epoche', ylabel='gewichteter MSE')  # Loss-Diagramm beschriften
achsen[0].legend()  # Kurven unterscheiden

gewichtsschritte = np.arange(len(next(iter(gewichtsverlauf.values()))))  # Ersten Updatezustand und Epochen positionieren
for name, werte_liste in gewichtsverlauf.items():  # Entwicklung jeder ausgewählten Verbindung zeichnen
    achsen[1].plot(gewichtsschritte, werte_liste, label=name)  # Gelerntes Gewicht über die Zeit darstellen
achsen[1].axhline(0.0, color='black', linewidth=0.8)  # Vorzeichenwechsel sichtbar machen
achsen[1].set(title='Aus Zufallsgewichten entstehen Signalsteuerungen', xlabel='Trainingsfortschritt', ylabel='Gewicht')  # Gewichtsplot beschriften
achsen[1].legend(fontsize=8)  # Fünf Gewichtspfade benennen
plt.tight_layout()  # Abstände beider Diagramme anpassen
plt.show()  # Trainingsverlauf ausgeben

print(f'Beste Epoche: {beste_epoche} von {EPOCHEN}')  # Modellauswahl dokumentieren
print(f'Bester Validierungs-Loss: {beste_validierung:.5f}')  # Unabhängigen Bestwert ausgeben
print(f'Trainingsdauer: {trainingsdauer:.1f} Sekunden')  # Laufzeit auf dem aktuellen Rechner ausgeben

## Funktioniert das nur mit großen Speicherwerten?

Nein. Die folgende Auswertung gruppiert die noch nie gesehenen Testsequenzen nach der Größe ihrer markierten Zahl. In vielen Sequenzen ist mindestens ein Ablenkungswert größer als die relevante Information. Entscheidend bleibt die Markierung, nicht der Betrag.

In [ ]:
test_vorhersagen = modell.predict(X_test, verbose=0)[:, :, 0]  # Vorhersage y_t an jedem Testzeitschritt berechnen
ausgabe_masken_test = X_test[:, :, 2].astype(bool)  # Tatsächliche Abfragepositionen auswählen
mae_abfragen = np.mean(np.abs(test_vorhersagen[ausgabe_masken_test] - y_test[:, :, 0][ausgabe_masken_test]))  # Fehler nur bei gewünschten Ausgaben bestimmen
mae_ohne_abfrage = np.mean(np.abs(test_vorhersagen[~ausgabe_masken_test]))  # Unerwünschte Ausgaben sollten nahe null bleiben
letzte_vorhersagen = test_vorhersagen[:, -1]  # Letzter Schritt ist bei jeder Sequenz eine Abfrage

gruppen = {  # Testziele nach ihrem Betrag gruppieren
    'klein\n|Ziel| < 0,2': np.abs(ziele_test) < 0.2,
    'mittel\n0,2–0,6': (np.abs(ziele_test) >= 0.2) & (np.abs(ziele_test) < 0.6),
    'groß\n|Ziel| ≥ 0,6': np.abs(ziele_test) >= 0.6,
}
gruppen_mae = [np.mean(np.abs(letzte_vorhersagen[maske] - ziele_test[maske])) for maske in gruppen.values()]  # MAE jeder Zielgröße berechnen

ablenkungsbetraege = np.abs(X_test[:, :, 0]).copy()  # Beträge aller Eingangswerte kopieren
ablenkungsbetraege[np.arange(ANZAHL_TEST), speicher_test] = -np.inf  # Markierte Zahl aus der Ablenkungssuche entfernen
groessere_ablenkung = np.max(ablenkungsbetraege, axis=1) > np.abs(ziele_test)  # Sequenzen mit größerem Distraktor erkennen
mae_mit_groesserer_ablenkung = np.mean(np.abs(letzte_vorhersagen[groessere_ablenkung] - ziele_test[groessere_ablenkung]))  # Schwierige Fälle getrennt bewerten

fig, achsen = plt.subplots(1, 2, figsize=(13, 4.5))  # Vorhersagen und Größengruppen vergleichen
achsen[0].scatter(ziele_test, letzte_vorhersagen, alpha=0.35, color='tab:blue')  # Wahre gegen vorhergesagte Speicherwerte zeichnen
achsen[0].plot([-1, 1], [-1, 1], color='black', linestyle=':', label='perfekte Ausgabe')  # Ideale Diagonale ergänzen
achsen[0].set(title='Noch nie gesehene Testsequenzen', xlabel='markierter Wert', ylabel='LSTM-Ausgabe am Ende')  # Scatterplot beschriften
achsen[0].legend()  # Referenzlinie benennen
achsen[1].bar(gruppen.keys(), gruppen_mae, color=['tab:green', 'tab:orange', 'tab:red'])  # MAE für kleine, mittlere und große Werte zeigen
achsen[1].set(title='Kleine Werte werden ebenfalls erinnert', ylabel='Test-MAE')  # Balkendiagramm beschriften
plt.tight_layout()  # Abstände anpassen
plt.show()  # Testauswertung ausgeben

print(f'MAE an allen Abfragepositionen: {mae_abfragen:.4f}')  # Hauptfehler des Gedächtnisses ausgeben
print(f'Mittlere |Ausgabe| ohne Abfrage: {mae_ohne_abfrage:.4f}')  # Geschlossenes Ausgabeerhalten bewerten
print(f'Anteil mit größerem Ablenkungswert: {groessere_ablenkung.mean():.1%}')  # Schwierigkeit des Datensatzes quantifizieren
print(f'MAE trotz größerem Ablenkungswert: {mae_mit_groesserer_ablenkung:.4f}')  # Markierungslernen bestätigen

## Gelernte Gewichte und vollständiger LSTM-Durchlauf

Keras speichert die vier LSTM-Blöcke in der Reihenfolge **Input-Signal $i_t$**, **Forget-Signal $f_t$**, **neue Information $g_t$** und **Output $o_t$**. Für jeden Block werden fünf Beiträge addiert: die drei Merkmale der Eingabe $x_t$, der vorherige Hidden State $h_{t-1}$ und der Bias.

Erst danach entstehen durch elementweise Multiplikation der **Behaltebeitrag** $f_t \odot c_{t-1}$ und der **Schreibbeitrag** $i_t \odot g_t$. Nun wird eine Testsequenz gewählt, deren markierter Wert klein ist und die trotzdem mindestens einen sehr großen Ablenkungswert enthält. Der komplette Keras-Durchlauf wird mit NumPy nachgerechnet.

Die folgenden Tabellen beantworten unterschiedliche Fragen:

1. **Gewichtstabelle:** Welche Eingaben beeinflussen die vier LSTM-Blöcke nach dem Training?
2. **Signal-Tabelle:** Was berechnet die LSTM in jedem Zeitschritt aus $x_t$ und $h_{t-1}$?
3. **Speicher-Tabelle:** Wie verändern Behalte- und Schreibbeitrag den Zellzustand $c_t$?
4. **Ausgabe:** Wie wird aus $c_t$ über $h_t$ schließlich die Vorhersage $y_t$?

Wichtig: Der zu merkende Wert muss im Zellzustand $c_t$ **nicht als identische Zahl** stehen. Die LSTM darf ihn intern anders codieren; der Dense-Layer übersetzt den sichtbaren Hidden State $h_t$ später wieder in die gewünschte Vorhersage.

In [ ]:
kernel, rekurrent, bias = lstm_schicht.get_weights()  # Gelernte LSTM-Gewichte auslesen
dense_kernel, dense_bias = ausgabe_schicht.get_weights()  # Gelernte Ausgabetransformation auslesen

gelernte_matrix = np.vstack([kernel, rekurrent, bias[None, :]])  # Alle Beiträge der vier Blöcke untereinander anordnen
gelernte_tabelle = pd.DataFrame(gelernte_matrix, index=['Wert in x_t', 'Speichermarker in x_t', 'Ausgabemarker in x_t', 'Hidden State h_{t-1}', 'Bias'], columns=block_namen)  # Gewichtstabelle mit den Namen der Folie beschriften
display(Markdown(
    '### Gelernte Gewichtstabelle: Worauf reagieren die vier Blöcke?\n\n'
    'Jede Zelle ist ein **gelerntes Gewicht**, noch kein Signalwert und kein gespeicherter Inhalt. Positive und negative Gewichte bestimmen gemeinsam mit den übrigen Beiträgen, ob $i_t$, $f_t$, $g_t$ oder $o_t$ in einem konkreten Zeitschritt groß oder klein wird.'
))
display(gelernte_tabelle.round(3))  # Tatsächlich gelernte Gewichte der vier LSTM-Blöcke anzeigen

vergleich_gewichte = []  # Zufällige Startwerte und gelernte Werte ausgewählter Pfade vergleichen
for name, (zeile, spalte) in wichtige_pfade.items():  # Fünf verständliche Pfade auswerten
    vergleich_gewichte.append({'Verbindung': name, 'Start': start_gewichte[0][zeile, spalte], 'gelernt': kernel[zeile, spalte]})  # Vorher und nachher sammeln
display(Markdown(
    '### Vorher–nachher: Welche Verbindungen hat das Training besonders verändert?\n\n'
    'Diese Auswahl vergleicht zufällige Startgewichte mit den gelernten Gewichten. Sie erklärt die entstandene Steuerung, aber noch nicht den Ablauf einer einzelnen Sequenz.'
))
display(pd.DataFrame(vergleich_gewichte).set_index('Verbindung').round(3))  # Lernergebnis kompakt darstellen

def sigmoid(z):  # Aktivierung für Input-Signal, Forget-Signal und Output definieren
    return 1.0 / (1.0 + np.exp(-z))  # Gewichtete Summe auf einen Wert zwischen null und eins abbilden

def protokolliere_lstm(eingabe_sequenz):  # Einen vollständigen Durchlauf mit den gelernten Gewichten nachrechnen
    h_alt = np.zeros(1, dtype=np.float32)  # Vorherigen Hidden State h_{t-1} vor dem ersten Schritt initialisieren
    c_alt = np.zeros(1, dtype=np.float32)  # Vorherigen Zellzustand c_{t-1} vor dem ersten Schritt initialisieren
    zeilen = []  # Alle Zwischenwerte sammeln
    for t, x_t in enumerate(eingabe_sequenz):  # Werte streng nacheinander verarbeiten
        z = x_t @ kernel + h_alt @ rekurrent + bias  # Alle Eingabe-, Kontext- und Bias-Beiträge addieren
        z_i, z_f, z_g, z_o = z  # Gemeinsame Keras-Matrix in vier Voraktivierungen zerlegen
        i_t, f_t, o_t = sigmoid(z_i), sigmoid(z_f), sigmoid(z_o)  # Input-Signal, Forget-Signal und Output berechnen
        g_t = np.tanh(z_g)  # Neue Information g_t berechnen
        behaltebeitrag_t = f_t * c_alt[0]  # Behaltebeitrag f_t · c_{t-1} berechnen
        schreibbeitrag_t = i_t * g_t  # Schreibbeitrag i_t · g_t berechnen
        c_t = behaltebeitrag_t + schreibbeitrag_t  # Neuen Zellzustand c_t aus beiden Beiträgen bilden
        h_t = o_t * np.tanh(c_t)  # Aktuellen Hidden State h_t erzeugen
        y_t = h_t * dense_kernel[0, 0] + dense_bias[0]  # Hidden State in die Vorhersage y_t umrechnen
        ereignis = 'SPEICHERN' if x_t[1] == 1 else ('AUSGEBEN' if x_t[2] == 1 else '–')  # Entscheidende Zeilen der Sequenz markieren
        zeilen.append({  # Eingaben, Signale, Beiträge, Zustände und Vorhersage protokollieren
            't': t, 'Ereignis': ereignis, 'x_t: Wert': x_t[0], 'x_t: Speichern': x_t[1], 'x_t: Ausgeben': x_t[2],
            'h_{t-1}': h_alt[0], 'c_{t-1}': c_alt[0], 'z_i': z_i, 'i_t (Input-Signal)': i_t,
            'z_f': z_f, 'f_t (Forget-Signal)': f_t, 'z_g': z_g, 'g_t (neue Info)': g_t,
            'Behaltebeitrag f_t·c_{t-1}': behaltebeitrag_t, 'Schreibbeitrag i_t·g_t': schreibbeitrag_t,
            'z_o': z_o, 'o_t (Output)': o_t, 'c_t': c_t, 'h_t': h_t, 'y_t (Vorhersage)': y_t,
        })
        h_alt = np.array([h_t], dtype=np.float32)  # h_t als h_{t-1} an den nächsten Schritt weitergeben
        c_alt = np.array([c_t], dtype=np.float32)  # c_t als c_{t-1} an den nächsten Schritt weitergeben
    return pd.DataFrame(zeilen)  # Vollständiges Rechenprotokoll zurückgeben

max_ablenkung_test = np.max(ablenkungsbetraege, axis=1)  # Größten Distraktor jeder Testsequenz bestimmen
kandidaten = np.flatnonzero((np.abs(ziele_test) >= 0.05) & (np.abs(ziele_test) <= 0.15) & (max_ablenkung_test > 0.85))  # Geeignetes kleines Speicherziel suchen
BEISPIEL = int(kandidaten[0])  # Erstes passendes, reproduzierbares Testbeispiel auswählen
beispiel_X = X_test[BEISPIEL]  # Eingabesequenz des Beispiels auslesen
beispiel_y = y_test[BEISPIEL, :, 0]  # Zielausgaben des Beispiels auslesen
zielwert_beispiel = float(ziele_test[BEISPIEL])  # Markierten und zu merkenden Wert benennen
speicher_schritt = int(speicher_test[BEISPIEL])  # Zeitschritt mit Speichermarker bestimmen
ausgabe_schritt = int(np.flatnonzero(beispiel_X[:, 2])[0])  # Zeitschritt mit Ausgabeanforderung bestimmen
protokoll = protokolliere_lstm(beispiel_X)  # Jeden LSTM-Schritt explizit berechnen
np.testing.assert_allclose(protokoll['y_t (Vorhersage)'], test_vorhersagen[BEISPIEL], atol=1e-6)  # NumPy-Rechnung gegen Keras prüfen

display(Markdown(
    '### Leseschlüssel für das konkrete Beispiel\n\n'
    f'- **Zu merkender Zielwert:** `{zielwert_beispiel:.3f}`. Er steht bei `t = {speicher_schritt}` in der Spalte **x_t: Wert**, während **x_t: Speichern = 1** ist.\n'
    f'- **Ausgabezeitpunkt:** `t = {ausgabe_schritt}`. Erst dort ist **x_t: Ausgeben = 1** und die Sollausgabe entspricht dem gemerkten Zielwert.\n'
    '- **Wo wird gespeichert?** Der Schreibbeitrag $i_t \odot g_t$ verändert den Zellzustand $c_t$. In den folgenden Schritten trägt der Behaltebeitrag $f_t \odot c_{t-1}$ diese interne Codierung weiter.\n'
    '- **Wie kommt die Information heraus?** $o_t$ macht einen Teil von $c_t$ als Hidden State $h_t$ sichtbar. Der Dense-Layer übersetzt $h_t$ anschließend in die Vorhersage $y_t$.'
))

signal_tabelle = protokoll[['t', 'Ereignis', 'x_t: Wert', 'x_t: Speichern', 'x_t: Ausgeben', 'h_{t-1}', 'i_t (Input-Signal)', 'f_t (Forget-Signal)', 'g_t (neue Info)', 'o_t (Output)']].copy()  # Eingabe, Kontext und vier berechnete Größen auswählen
display(Markdown(
    '### Signal-Tabelle: Was entscheidet die LSTM in jedem Zeitschritt?\n\n'
    'Jede Zeile ist ein Zeitschritt. **SPEICHERN** markiert die Zeile, in der der Zielwert aufgenommen werden soll; **AUSGEBEN** markiert die spätere Abfrage. Die vier rechten Spalten zeigen die berechneten Signale beziehungsweise die neue Information – noch nicht den endgültigen Speicherinhalt.'
))
display(signal_tabelle.round(3))  # Eingabe x_t, vorherigen Hidden State und alle Signale anzeigen

zustands_tabelle = protokoll[['t', 'Ereignis', 'c_{t-1}', 'Behaltebeitrag f_t·c_{t-1}', 'Schreibbeitrag i_t·g_t', 'c_t', 'o_t (Output)', 'h_t', 'y_t (Vorhersage)']].copy()  # Speicher- und Ausgabepfad getrennt anordnen
zustands_tabelle['Sollausgabe an t'] = beispiel_y  # Gewünschte Ausgabe pro Zeitschritt ergänzen
display(Markdown(
    '### Speicher- und Ausgabe-Tabelle: Wo bleibt die Information und wie kommt sie heraus?\n\n'
    '**c_t ist der interne Speicher.** Er entsteht aus Behaltebeitrag plus Schreibbeitrag. **h_t ist die nach außen sichtbare interne Darstellung**, die durch $o_t$ gesteuert wird. **y_t ist erst die Vorhersage des gesamten Modells.** Die Sollausgabe ist bis zur Abfrage null; erst in der AUSGEBEN-Zeile soll der zuvor markierte Zielwert erscheinen.'
))
display(zustands_tabelle.round(3))  # Zweite Hälfte des Rechenwegs mit den Namen der Folie anzeigen

speicher_zeile = protokoll.iloc[speicher_schritt]  # Werte der markierten Speicherzeile für die Interpretation auslesen
ausgabe_zeile = protokoll.iloc[ausgabe_schritt]  # Werte der markierten Ausgabezeile für die Interpretation auslesen
display(Markdown(
    '### So liest du die beiden entscheidenden Zeilen\n\n'
    f'- **SPEICHERN bei t = {speicher_schritt}:** Der Zielwert `{zielwert_beispiel:.3f}` wird nicht eins zu eins kopiert. Der Schreibbeitrag beträgt `{speicher_zeile["Schreibbeitrag i_t·g_t"]:.3f}` und verändert den internen Zellzustand von `{speicher_zeile["c_{t-1}"]:.3f}` auf `{speicher_zeile["c_t"]:.3f}`. Darin liegt nun eine gelernte **Codierung** der Information.\n'
    f'- **AUSGEBEN bei t = {ausgabe_schritt}:** Der Output beträgt `{ausgabe_zeile["o_t (Output)"]:.3f}`. Dadurch entsteht $h_t = {ausgabe_zeile["h_t"]:.3f}$; der Dense-Layer decodiert daraus $y_t = {ausgabe_zeile["y_t (Vorhersage)"]:.3f}$. Die geforderte Sollausgabe ist `{beispiel_y[ausgabe_schritt]:.3f}`.'
))

t = np.arange(SEQUENZLAENGE)  # Zeitschritte für alle Teilplots vorbereiten
fig, achsen = plt.subplots(5, 1, figsize=(12, 13.5), sharex=True)  # Eingabe, Signale, Beiträge, Zustände und Vorhersage untereinander zeichnen
achsen[0].plot(t, beispiel_X[:, 0], 'o-', color='0.45', label='Eingabe x_t (Wert)')  # Alle Werte des Testbeispiels zeichnen
achsen[0].scatter(speicher_test[BEISPIEL], ziele_test[BEISPIEL], s=150, color='tab:blue', label='markierter kleiner Wert', zorder=4)  # Zu merkende Zahl hervorheben
achsen[0].set_ylabel('Eingabe')  # Erste Achse beschriften
achsen[0].legend(loc='best')  # Markierung erklären
achsen[1].plot(t, protokoll['i_t (Input-Signal)'], 'o-', label='Input-Signal i_t')  # Input-Signal darstellen
achsen[1].plot(t, protokoll['f_t (Forget-Signal)'], 's-', label='Forget-Signal f_t')  # Forget-Signal darstellen
achsen[1].plot(t, protokoll['g_t (neue Info)'], 'D-', label='neue Information g_t')  # Neue Information darstellen
achsen[1].plot(t, protokoll['o_t (Output)'], '^-', label='Output o_t')  # Output darstellen
achsen[1].set(ylim=(-1.05, 1.05), ylabel='Signale')  # Sigmoid-Signale und tanh-Information gemeinsam skalieren
achsen[1].legend(loc='best', ncols=2)  # Vier Größen unterscheiden
achsen[2].plot(t, protokoll['Behaltebeitrag f_t·c_{t-1}'], 'o-', color='tab:blue', label='Behaltebeitrag f_t · c_{t-1}')  # Weitergegebenen alten Zellzustand zeichnen
achsen[2].plot(t, protokoll['Schreibbeitrag i_t·g_t'], 's-', color='tab:red', label='Schreibbeitrag i_t · g_t')  # Neu geschriebenen Beitrag zeichnen
achsen[2].set_ylabel('Beiträge')  # Produktachse beschriften
achsen[2].legend(loc='best')  # Behalte- und Schreibbeitrag unterscheiden
achsen[3].plot(t, protokoll['c_t'], 'o-', color='tab:purple', label='Zellzustand c_t')  # Internes Gedächtnis zeichnen
achsen[3].plot(t, protokoll['h_t'], 's-', color='tab:green', label='Hidden State h_t')  # Sichtbaren State zeichnen
achsen[3].set_ylabel('Zustand')  # Zustandsachse beschriften
achsen[3].legend(loc='best')  # Zustände unterscheiden
achsen[4].plot(t, protokoll['y_t (Vorhersage)'], 'o-', color='tab:orange', label='Vorhersage y_t')  # Vorhersage an jedem Schritt zeichnen
achsen[4].scatter(t[beispiel_X[:, 2].astype(bool)], beispiel_y[beispiel_X[:, 2].astype(bool)], marker='X', s=110, color='tab:green', label='geforderte Ausgabe')  # Richtige Abfragewerte markieren
achsen[4].set(xlabel='Zeitschritt', ylabel='Ausgabe')  # Letzte Achse beschriften
achsen[4].set_xticks(t)  # Jeden Zeitschritt anzeigen
achsen[4].legend(loc='best')  # Vorhersage und Ziel unterscheiden
fig.suptitle(f'Kleines Ziel {ziele_test[BEISPIEL]:.3f}, größter Ablenkungswert {max_ablenkung_test[BEISPIEL]:.3f}')  # Schwierigkeit des Beispiels nennen
plt.tight_layout()  # Abstände anpassen
plt.show()  # Vollständigen Testdurchlauf visualisieren

print('NumPy-Loop und Keras-Ausgabe stimmen bis auf Rundungsfehler überein.')  # Erfolgreiche Kontrolle bestätigen

## Zwei Zeitschritte vollständig mit Zahlen ausrechnen

Die nächste Zelle setzt die gelernten Gewichte tatsächlich in jede Formel ein. Zuerst wird der Speicherzeitpunkt berechnet, danach die Ausgabeanforderung. So ist sichtbar, wie aus $x_t$ und $h_{t-1}$ die vier Größen $i_t$, $f_t$, $g_t$ und $o_t$ entstehen. Anschließend werden **Behaltebeitrag** $f_t \odot c_{t-1}$ und **Schreibbeitrag** $i_t \odot g_t$ getrennt ausgerechnet, bevor daraus $c_t$, $h_t$ und die Vorhersage $y_t$ entstehen.

Der **Zielwert** ist die Zahl, die bei gesetztem Speichermarker aufgenommen werden soll. Die **Sollausgabe** ist dagegen die gewünschte Modellausgabe im jeweiligen Zeitschritt: Beim Speichern ist sie noch null, und erst beim späteren Ausgabemarker soll sie dem Zielwert entsprechen.

In [ ]:
def erklaere_schritt(schritt):  # Einen einzelnen LSTM-Schritt als vollständige Zahlenrechnung ausgeben
    zeile = protokoll.iloc[schritt]  # Bereits kontrollierte Zwischenwerte dieses Schritts auslesen
    x_t = beispiel_X[schritt]  # Drei aktuelle Eingabemerkmale bereitstellen
    h_alt = zeile['h_{t-1}']  # Vorherigen Hidden State h_{t-1} auslesen
    c_alt = zeile['c_{t-1}']  # Vorherigen Zellzustand c_{t-1} auslesen
    blocke = [  # Namen, Matrixspalten, Aktivierungen und Ergebnisnamen definieren
        ('Input-Signal i_t', 0, 'Sigmoid', 'i_t (Input-Signal)'),
        ('Forget-Signal f_t', 1, 'Sigmoid', 'f_t (Forget-Signal)'),
        ('Neue Information g_t', 2, 'tanh', 'g_t (neue Info)'),
        ('Output o_t', 3, 'Sigmoid', 'o_t (Output)'),
    ]
    print(f'\n===== Zeitschritt {schritt} =====')  # Beginn der Detailrechnung markieren
    print(f'x_t = [Wert {x_t[0]:.3f}, Speichern {x_t[1]:.0f}, Ausgeben {x_t[2]:.0f}]')  # Aktuelle Eingabe x_t nennen
    print(f'h_{{t-1}} = {h_alt:.3f}, c_{{t-1}} = {c_alt:.3f}\n')  # Vorherige Zustände nennen

    for name, spalte, aktivierung, ergebnis_name in blocke:  # Alle vier linearen Blöcke einzeln berechnen
        beitraege = [x_t[j] * kernel[j, spalte] for j in range(3)]  # Drei Eingabebeiträge berechnen
        hidden_beitrag = h_alt * rekurrent[0, spalte]  # Beitrag des vorherigen Hidden States berechnen
        summe = sum(beitraege) + hidden_beitrag + bias[spalte]  # Voraktivierung vollständig addieren
        print(f'{name}:')  # Aktuellen Block benennen
        print(f'  z = ({x_t[0]:.3f}·{kernel[0, spalte]:.3f}) + ({x_t[1]:.0f}·{kernel[1, spalte]:.3f}) + ({x_t[2]:.0f}·{kernel[2, spalte]:.3f})')  # Eingabemultiplikationen zeigen
        print(f'      + ({h_alt:.3f}·{rekurrent[0, spalte]:.3f}) + {bias[spalte]:.3f} = {summe:.3f}')  # Kontext und Bias ergänzen
        print(f'  {ergebnis_name} = {aktivierung}({summe:.3f}) = {zeile[ergebnis_name]:.3f}\n')  # Aktivierung und Blockergebnis zeigen

    print(f'Behaltebeitrag: f_t·c_{{t-1}} = {zeile["f_t (Forget-Signal)"]:.3f}·{c_alt:.3f} = {zeile["Behaltebeitrag f_t·c_{t-1}"]:.3f}')  # Forget-Signal mit vorherigem Zellzustand multiplizieren
    print(f'Schreibbeitrag: i_t·g_t = {zeile["i_t (Input-Signal)"]:.3f}·{zeile["g_t (neue Info)"]:.3f} = {zeile["Schreibbeitrag i_t·g_t"]:.3f}')  # Input-Signal mit neuer Information multiplizieren
    print(f'Zellzustand: c_t = Behaltebeitrag + Schreibbeitrag = {zeile["Behaltebeitrag f_t·c_{t-1}"]:.3f} + {zeile["Schreibbeitrag i_t·g_t"]:.3f} = {zeile["c_t"]:.3f}')  # Speicherupdate vollständig einsetzen
    print(f'Hidden State: h_t = o_t·tanh(c_t) = {zeile["o_t (Output)"]:.3f}·tanh({zeile["c_t"]:.3f}) = {zeile["h_t"]:.3f}')  # Sichtbaren State berechnen
    print(f'Vorhersage: y_t = h_t·w + b = {zeile["h_t"]:.3f}·{dense_kernel[0, 0]:.3f} + {dense_bias[0]:.3f} = {zeile["y_t (Vorhersage)"]:.3f}')  # Finale Vorhersage berechnen
    print(f'Zu merkender Zielwert der Sequenz: {zielwert_beispiel:.3f}')  # Inhalt nennen, der intern erhalten werden soll
    print(f'Sollausgabe an diesem Schritt: {beispiel_y[schritt]:.3f}')  # Gewünschte Ausgabe dieses Zeitschritts nennen
    if x_t[1] == 1:  # Bedeutung des Speicherzeitpunkts hervorheben
        print('Bedeutung: Der Zielwert wird jetzt intern in c_t codiert; ausgegeben werden soll er noch nicht.')
    if x_t[2] == 1:  # Bedeutung des Ausgabezeitpunkts hervorheben
        print('Bedeutung: Jetzt soll die gespeicherte Information über h_t als y_t herauskommen.')

erklaere_schritt(speicher_schritt)  # Vollständige Rechnung beim Schreiben in den Speicher ausgeben
erklaere_schritt(ausgabe_schritt)  # Vollständige Rechnung beim Lesen aus dem Speicher ausgeben

## Was bedeutet ein Sigmoid-Signal nahe 1 konkret?

Nicht der rohe Eingabewert wird direkt mit einer festen Schwelle verglichen. Erst die gelernte Summe $z$ entscheidet:

- $z=-6$ ergibt ungefähr 0,002: Das Signal lässt fast nichts passieren,
- $z=0$ ergibt 0,5: Das Signal lässt etwa die Hälfte passieren,
- $z=6$ ergibt ungefähr 0,998: Das Signal lässt fast alles passieren.

Ein kleines Speicherziel kann deshalb problemlos verarbeitet werden. Die LSTM darf den Zahlenwert über die neue Information $g_t$, das Input-Signal $i_t$ oder über beide Wege codieren. Erst ihr Produkt, der Schreibbeitrag $i_t \odot g_t$, gelangt in den Zellzustand. Welche Zerlegung die LSTM wählt, ergibt sich aus dem Training und ist erst in der vollständigen Zahlenrechnung sichtbar.

In [ ]:
testwerte = [-0.08, 0.08, 0.80]  # Sehr kleine negative, kleine positive und große Information vergleichen
experiment = []  # Ergebnisse des kontrollierten Eingabeexperiments sammeln
for neuer_wert in testwerte:  # Nur die markierte Zahl verändern, alle Ablenkungen unverändert lassen
    variante = beispiel_X.copy()  # Dieselbe Testsequenz kopieren
    variante[speicher_schritt, 0] = neuer_wert  # Markierten Speicherinhalt gezielt ersetzen
    variante_protokoll = protokolliere_lstm(variante)  # Gelernte LSTM erneut vollständig berechnen
    speicher_zeile = variante_protokoll.iloc[speicher_schritt]  # Signale und Beiträge am Speicherzeitpunkt auslesen
    experiment.append({
        'zu speichernder Wert': neuer_wert,
        'Input-Signal i_t': speicher_zeile['i_t (Input-Signal)'],
        'Forget-Signal f_t': speicher_zeile['f_t (Forget-Signal)'],
        'Neue Information g_t': speicher_zeile['g_t (neue Info)'],
        'Behaltebeitrag f_t·c_{t-1}': speicher_zeile['Behaltebeitrag f_t·c_{t-1}'],
        'Schreibbeitrag i_t·g_t': speicher_zeile['Schreibbeitrag i_t·g_t'],
        'Zellzustand c_t nach Speichern': speicher_zeile['c_t'],
        'Vorhersage y_t am letzten Schritt': variante_protokoll.iloc[-1]['y_t (Vorhersage)'],
    })
experiment_tabelle = pd.DataFrame(experiment)  # Drei Größenvarianten tabellarisch bündeln
experiment_tabelle['absoluter Fehler am Ende'] = np.abs(experiment_tabelle['Vorhersage y_t am letzten Schritt'] - experiment_tabelle['zu speichernder Wert'])  # Abweichung zwischen Zielwert und späterer Vorhersage berechnen
display(Markdown(
    '### Kontrolltabelle: Wird wirklich der markierte Wert gespeichert?\n\n'
    'In jeder Zeile bleibt die gesamte Sequenz gleich; nur der markierte, zu speichernde Wert wird ersetzt. Die mittleren Spalten zeigen, wie dieser Wert intern verarbeitet wird. Entscheidend sind die erste und die letzte Spalte: Die spätere Vorhersage soll zum zu speichernden Wert passen, und der absolute Fehler sollte klein sein.'
))
display(experiment_tabelle.round(3))  # Zeigen, dass dieselbe Signal- und Beitragslogik kleine und große Inhalte verarbeitet

z_kurve = np.linspace(-8, 8, 400)  # Mögliche Voraktivierungen für die Sigmoid-Kurve erzeugen
fig, achsen = plt.subplots(1, 2, figsize=(13, 4.5))  # Größenexperiment und Sigmoid-Funktion nebeneinander darstellen
achsen[0].plot(testwerte, testwerte, color='black', linestyle=':', label='perfekte Erinnerung')  # Sollausgaben als Referenz zeichnen
achsen[0].scatter(testwerte, experiment_tabelle['Vorhersage y_t am letzten Schritt'], s=100, color='tab:blue', label='Vorhersage y_t')  # Gelernte Vorhersagen zeigen
achsen[0].set(title='Gleiche Ablenkungen, anderer Speicherwert', xlabel='zu speichernder Wert', ylabel='Vorhersage y_t am Ende')  # Experiment beschriften
achsen[0].legend()  # Referenz und Modell unterscheiden

achsen[1].plot(z_kurve, sigmoid(z_kurve), color='tab:purple', linewidth=2)  # Sigmoid-Funktion der Signale zeichnen
marken_z = np.array([-6.0, 0.0, 6.0])  # Drei anschauliche Voraktivierungen auswählen
achsen[1].scatter(marken_z, sigmoid(marken_z), s=80, color='tab:orange', zorder=3)  # Fast geschlossen, halb und fast offen markieren
for z_wert in marken_z:  # Numerischen Signalwert an jede Markierung schreiben
    achsen[1].annotate(f'{sigmoid(z_wert):.3f}', (z_wert, sigmoid(z_wert)), xytext=(5, -15), textcoords='offset points')  # Wert lesbar platzieren
achsen[1].set(title='Sigmoid übersetzt die gelernte Summe z', xlabel='gewichtete Summe z', ylabel='Signalwert')  # Sigmoid-Kurve beschriften
plt.tight_layout()  # Abstände beider Diagramme anpassen
plt.show()  # Abschließendes Experiment ausgeben

## Kerngedanke

Die LSTM lernt keine Liste „großer Wert = merken“. Der Fehler der gewünschten Vorhersage $y_t$ wird durch alle Zeitschritte zurückpropagiert. Dadurch verändern sich die Gewichte, die aus der **Eingabe $x_t$**, dem **vorherigen Hidden State $h_{t-1}$** und dem **Bias** die Summen für $i_t$, $f_t$, $g_t$ und $o_t$ bilden.

In diesem konkreten Trainingslauf senkt der Speichermarker vor allem das Forget-Signal $f_t$ und beeinflusst die neue Information $g_t$. Der Zahlenwert steuert besonders das Input-Signal $i_t$. In den Zellzustand gelangt jedoch erst der **Schreibbeitrag** $i_t \odot g_t$; vom vorherigen Zellzustand $c_{t-1}$ bleibt der **Behaltebeitrag** $f_t \odot c_{t-1}$. Der Ausgabemarker erhöht am Ende den Output $o_t$. Eine andere Initialisierung kann dieselbe Aufgabe mit einer etwas anderen Aufteilung lösen – entscheidend ist immer die vollständige Rechnung. In realen Aufgaben gibt es meist keine fertigen Marker; dort muss die LSTM relevante Muster aus Wörtern, Messwerten oder anderen Kontextmerkmalen selbst lernen.